In [0]:
# XSD PARSER UTILITY
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

import xml.etree.ElementTree as ET

def extract_metadata(xsd_path):
    # Parse XSD
    tree = ET.parse(xsd_path)
    # Get root
    root = tree.getroot()
    # Namespace
    ns = {
        'xs': 'http://www.w3.org/2001/XMLSchema'
    }

    # Extract all XSD elements
    elements = root.findall(".//xs:element", ns)
    metadata = []

    for elem in elements:
        # Skip container elements
        child_elements = elem.findall(
            ".//xs:element",
            ns
        )

        if len(child_elements) > 0:
            continue

        column_name = elem.attrib.get("name")

        data_type = elem.attrib.get(
            "type",
            "xs:string"
        )

        min_occurs = elem.attrib.get(
            "minOccurs",
            "1"
        )

        max_occurs = elem.attrib.get(
            "maxOccurs",
            "1"
        )

        metadata.append({
            "column_name": column_name,
            "xsd_type": data_type,
            "min_occurs": min_occurs,
            "max_occurs": max_occurs,
            "parent": None,
            "path": column_name,
            "level": 0
        })

    schema = StructType([
        StructField("column_name", StringType(),True),
        StructField("xsd_type",StringType(),True),
        StructField("min_occurs",StringType(),True),
        StructField("max_occurs",StringType(),True),
        StructField("parent",StringType(),True),
        StructField("path",StringType(),True),
        StructField("level",IntegerType(),True)
    ])

    metadata_df = spark.createDataFrame(
        metadata,
        schema=schema
    )
    return metadata_df

In [0]:

# DATATYPE MAPPER UTILITY

from pyspark.sql.functions import (
    col,
    when
)

def map_xsd_to_sql(metadata_df):

    mapped_df = metadata_df.withColumn(

        "sql_type",
        when(
            col("xsd_type") == "xs:string",
            "STRING"
        )
        .when(
            col("xsd_type") == "xs:int",
            "INT"
        )
        .when(
            col("xsd_type") == "xs:integer",
            "INT"
        )
        .when(
            col("xsd_type") == "xs:decimal",
            "DOUBLE"
        )
        .when(
            col("xsd_type") == "xs:double",
            "DOUBLE"
        )
        .when(
            col("xsd_type") == "xs:float",
            "FLOAT"
        )
        .when(
            col("xsd_type") == "xs:boolean",
            "BOOLEAN"
        )
        .when(
            col("xsd_type") == "xs:date",
            "DATE"
        )
        .when(
            col("xsd_type") == "xs:dateTime",
            "TIMESTAMP"
        )
        .otherwise("STRING")
    )

    return mapped_df

In [0]:

# SCHEMA REGISTRY UTILITY

from pyspark.sql.functions import (
    current_timestamp,
    lit
)


def enrich_registry_metadata(
    metadata_df,
    xsd_file,
    root_element,
    target_table,
    schema_version=1
):

    metadata_df = metadata_df.withColumn(
        "xsd_file",
        lit(xsd_file)
    )

    metadata_df = metadata_df.withColumn(
        "root_element",
        lit(root_element)
    )

    metadata_df = metadata_df.withColumn(
        "target_table",
        lit(target_table)
    )

    metadata_df = metadata_df.withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )

    metadata_df = metadata_df.withColumn(
        "schema_version",
        lit(schema_version)
    )

    return metadata_df


def write_to_registry(
    metadata_df,
    registry_table
):

    metadata_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(registry_table)


def read_registry(
    registry_table
):

    return spark.table(registry_table)